<a href="https://colab.research.google.com/github/furkanaras0/Tez-recipe-recommender-system/blob/main/FoodcomTFRS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. KURULUM VE KÜTÜPHANELER
# ==========================================
!pip install -q tensorflow-recommenders tf-keras sentence-transformers

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" # Keras 2 uyumluluğu

from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/Tez/food-com-recipes-and-user-interactions.zip -d /content/

import numpy as np
import pandas as pd
import ast
import scipy.sparse as sp
import tensorflow as tf
import tensorflow_recommenders as tfrs
import pickle
import gc

from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
from sklearn.model_selection import train_test_split

DATA_PATH = "/content"
print("✅ TFRS, SBERT ve gerekli tüm kütüphaneler yüklendi.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 4.7 MB/s eta 0:00:00
Mounted at /content/drive
✅ TFRS, SBERT ve gerekli tüm kütüphaneler yüklendi.


In [ ]:
# ==========================================
# 2. VERİ YÜKLEME VE %80 TRAIN - %20 TEST AYRIMI (MÜKEMMELLEŞTİRİLMİŞ)
# ==========================================


print("🔄 Veriler yükleniyor ve bellek optimizasyonu ile temizleniyor...")

interactions = pd.read_csv(DATA_PATH + "/RAW_interactions.csv",
                           usecols=["user_id", "recipe_id", "rating"],
                           dtype={"user_id": "int32", "recipe_id": "int32", "rating": "float32"})
recipes = pd.read_csv(DATA_PATH + "/RAW_recipes.csv",
                      dtype={"id": "int32"}, engine='python', on_bad_lines='skip')

# 1. Temel Temizlik
interactions = interactions[interactions["rating"] > 0]
interactions = interactions.drop_duplicates(subset=["user_id", "recipe_id"], keep="last").dropna()
recipes = recipes.dropna(subset=['id', 'name'])

valid_recipe_ids = set(recipes["id"].unique())
interactions = interactions[interactions["recipe_id"].isin(valid_recipe_ids)]

# 2. MÜKEMMELLEŞTİRME: Çöp Veriyi (Gürültüyü) Temizleme
# Sadece 1 kez yenmiş "izole" tarifleri atıyoruz ki model aradaki işbirliğini öğrenebilsin.
item_counts = interactions['recipe_id'].value_counts()
active_items = item_counts[item_counts >= 2].index
interactions = interactions[interactions['recipe_id'].isin(active_items)]

# En az 3 etkileşimi olan kullanıcılar
user_counts = interactions['user_id'].value_counts()
active_users = user_counts[user_counts >= 3].index
interactions_filtered = interactions[interactions['user_id'].isin(active_users)].copy()

# 3. MÜKEMMELLEŞTİRME: Bellek Yönetimi
# A100 GPU kullansak bile CPU RAM'ini rahatlatmak hız kazandırır.
del interactions
gc.collect() # İşletim sistemine "Kullanılmayan verileri çöpe at" emri

# 4. Train / Test Ayrımı
train_df, test_df = train_test_split(interactions_filtered, test_size=0.20, random_state=42, shuffle=True)

# 5. Kusursuz Cold-Start Koruması
train_users = set(train_df["user_id"].unique())
train_items = set(train_df["recipe_id"].unique())

test_df = test_df[test_df["user_id"].isin(train_users) & test_df["recipe_id"].isin(train_items)].reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

# Ana tarif veri setini de süzülmüş train setine göre daraltıyoruz
recipes = recipes[recipes["id"].isin(train_items)].reset_index(drop=True)

print(f"✅ Eğitim Seti: {len(train_df)} etkileşim | Test Seti: {len(test_df)} etkileşim")
print(f"📊 Modelin Önündeki Temiz Veri: {len(train_users)} Kullanıcı | {len(train_items)} Benzersiz Tarif")

🔄 Veriler yükleniyor ve bellek optimizasyonu ile temizleniyor...
✅ Eğitim Seti: 646327 etkileşim | Test Seti: 154843 etkileşim
📊 Modelin Önündeki Temiz Veri: 33565 Kullanıcı | 128083 Benzersiz Tarif


In [ ]:
# ==========================================
# 3. ÖZELLİK ÇIKARIMI (SBERT VE TF-IDF) - MÜKEMMELLEŞTİRİLMİŞ
# ==========================================


print("🔄 Özellikler işleniyor ve anlamsal yapılar (Context) kuruluyor...")

def safe_literal_eval(x):
    if pd.isna(x) or x == "": return []
    try: return x if isinstance(x, list) else ast.literal_eval(x)
    except: return []

recipes["tags"] = recipes["tags"].apply(safe_literal_eval)
recipes["ingredients"] = recipes["ingredients"].apply(safe_literal_eval)
recipes["nutrition"] = recipes["nutrition"].apply(safe_literal_eval)

recipes["minutes"] = pd.to_numeric(recipes["minutes"], errors="coerce").fillna(0).clip(lower=0)
recipes["time_bucket"] = pd.cut(recipes["minutes"], bins=[-1, 15, 30, 60, float("inf")], labels=["very_fast", "fast", "medium", "long"]).astype(str)

recipes["calories"] = recipes["nutrition"].apply(lambda x: float(x[0]) if isinstance(x, list) and len(x)>0 else 0.0)
recipes["calorie_bucket"] = pd.cut(recipes["calories"], bins=[-1, 200, 500, float("inf")], labels=["low_cal", "medium_cal", "high_cal"]).astype(str)

# --- MÜKEMMELLEŞTİRME: Anlamsal Metin İnşası (Semantic Context Structuring) ---
# SBERT'in daha iyi anlaması için kelimeleri yığmak yerine cümle benzeri bir yapı kuruyoruz.
# Ayrıca süre ve kalori bilgilerini de metne yediriyoruz.
print("📝 SBERT için tarif metinleri zenginleştiriliyor...")

names = recipes["name"].fillna("Unknown Recipe").tolist()
tags_list = recipes["tags"].apply(lambda x: ", ".join(x)).tolist()
ingredients_list = recipes["ingredients"].apply(lambda x: ", ".join(x)).tolist()
times = recipes["time_bucket"].tolist()
calories = recipes["calorie_bucket"].tolist()

# Hızlı list comprehension ile birleştirme
recipes["content_text"] = [
    f"Recipe: {n}. Features: {t} time, {c} calories. Ingredients: {i}. Tags: {tg}."
    for n, t, c, i, tg in zip(names, times, calories, ingredients_list, tags_list)
]

# 1. METRİKLER İÇİN: TF-IDF (Sadece hesaplama için, modele GİRMEYECEK)
print("📊 Değerlendirme metrikleri için TF-IDF matrisi oluşturuluyor...")
tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(recipes['content_text'])
recipe_id_to_tfidf_idx = {str(rid): i for i, rid in enumerate(recipes['id'])}

# 2. MODEL İÇİN: SBERT (Sentence-BERT) Zekası (Modele GİRECEK)
print(f"\n🧠 SBERT Semantik Vektörleri Çıkarılıyor (A100 GPU aktif)...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# A100 GPU'nun hakkını vermek için batch_size'ı yüksek tutuyoruz (Hızlandırır)
sbert_embeddings = sbert_model.encode(recipes['content_text'].tolist(), batch_size=512, show_progress_bar=True)

# Hızlı sözlük oluşturma
recipe_ids_str = recipes['id'].astype(str).tolist()
recipe_id_to_sbert = dict(zip(recipe_ids_str, sbert_embeddings))

print(f"✅ SBERT vektörleşmesi tamamlandı! (Boyut: {sbert_embeddings.shape})")

🔄 Özellikler işleniyor ve anlamsal yapılar (Context) kuruluyor...
📝 SBERT için tarif metinleri zenginleştiriliyor...
📊 Değerlendirme metrikleri için TF-IDF matrisi oluşturuluyor...

🧠 SBERT Semantik Vektörleri Çıkarılıyor (A100 GPU aktif)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/251 [00:00<?, ?it/s]

✅ SBERT vektörleşmesi tamamlandı! (Boyut: (128083, 384))


In [ ]:
# ==========================================
# 4. TFRS TENSOR VERİ SETİ
# ==========================================

print("🔄 tf.data.Dataset oluşturuluyor (A100 Veri Hattı Optimizasyonu ile)...")

# ID'leri String'e çeviriyoruz (TFRS için zorunlu)
train_df["user_id"] = train_df["user_id"].astype(str)
train_df["recipe_id"] = train_df["recipe_id"].astype(str)
test_df["user_id"] = test_df["user_id"].astype(str)
test_df["recipe_id"] = test_df["recipe_id"].astype(str)
recipes["id"] = recipes["id"].astype(str)

# Eğitim verisi ile tarif özelliklerini birleştiriyoruz
train_merged = train_df.merge(recipes[['id', 'time_bucket', 'calorie_bucket']], left_on='recipe_id', right_on='id')

# Train için SBERT vektörlerini hizala
train_sbert_vectors = np.array([recipe_id_to_sbert[rid] for rid in train_merged["recipe_id"].values])

# Adaylar (Candidates) için SBERT hizalaması
all_sbert_vectors = np.array([recipe_id_to_sbert[rid] for rid in recipes["id"].values])

# ---------------------------------------------------------
# 🚀 MÜKEMMELLEŞTİRME: tf.data.AUTOTUNE ile Veri Hattı Hızlandırma
# A100 GPU veriyi çok hızlı işler, CPU'nun ona veri yetiştirmesi için prefetch kullanıyoruz.
BATCH_SIZE = 2048 # A100 için ideal, devasa lokma boyutu

train_ds = tf.data.Dataset.from_tensor_slices({
    "user_id": train_merged["user_id"].values,
    "recipe_id": train_merged["recipe_id"].values,
    "time_bucket": train_merged["time_bucket"].values,
    "calorie_bucket": train_merged["calorie_bucket"].values,
    "sbert_vector": train_sbert_vectors
}).shuffle(100000).batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

recipes_ds = tf.data.Dataset.from_tensor_slices({
    "recipe_id": recipes["id"].values,
    "time_bucket": recipes["time_bucket"].values,
    "calorie_bucket": recipes["calorie_bucket"].values,
    "sbert_vector": all_sbert_vectors
}).batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

# ---------------------------------------------------------
# 🚨 İŞTE HATAYI ÇÖZEN O KRİTİK DEĞİŞKENLER BURADA:
unique_user_ids = train_df["user_id"].unique()
unique_recipe_ids = recipes["id"].unique()
unique_time_buckets = recipes["time_bucket"].unique()
unique_cal_buckets = recipes["calorie_bucket"].unique()

# ---------------------------------------------------------
# 🧹 MÜKEMMELLEŞTİRME: Sistem RAM'ini Temizleme
# Tensor veri setleri oluşturulduktan sonra geçici numpy dizilerine ihtiyacımız yok.
del train_merged, train_sbert_vectors, all_sbert_vectors
gc.collect()

print(f"✅ Tensor veri setleri hazır! ({len(unique_user_ids)} Kullanıcı, {len(unique_recipe_ids)} Tarif)")

🔄 tf.data.Dataset oluşturuluyor (A100 Veri Hattı Optimizasyonu ile)...
✅ Tensor veri setleri hazır! (33565 Kullanıcı, 128083 Tarif)


In [ ]:
# ==========================================
# 5. SBERT DESTEKLİ TFRS MODEL MİMARİSİ
# ==========================================
print("🧠 Gelişmiş SBERT Destekli TFRS İki Kuleli Model Kuruluyor...")

# Nihai Vektör Uzayı (Hem kullanıcı hem tarif bu uzayda buluşacak)
EMBEDDING_DIM = 128

class UserModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        # Kullanıcı ID'lerini vektöre çeviriyoruz
        self.user_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_user_ids, mask_token=None),
            tf.keras.layers.Embedding(len(unique_user_ids) + 1, EMBEDDING_DIM)
        ])

        # MÜKEMMELLEŞTİRME: Kullanıcı kulesine de derinlik (zekâ) kattık
        self.user_dense = tf.keras.Sequential([
            tf.keras.layers.Dense(EMBEDDING_DIM, activation='relu'),
            tf.keras.layers.Dense(EMBEDDING_DIM) # Son katmanda aktivasyon yok (L2 norm'a bırakıyoruz)
        ])

    def call(self, inputs):
        # Önce lookup, sonra sinir ağı, en son yön bulma (L2)
        x = self.user_embedding(inputs)
        x = self.user_dense(x)
        return tf.math.l2_normalize(x, axis=1)

class RecipeModel(tf.keras.Model):
    def __init__(self):
        super().__init__()

        # --- KATEGORİK BİLGİLER (EMBEDDINGS) ---
        self.id_emb = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_recipe_ids, mask_token=None),
            tf.keras.layers.Embedding(len(unique_recipe_ids) + 1, 64)
        ])
        self.time_emb = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_time_buckets, mask_token=None),
            tf.keras.layers.Embedding(len(unique_time_buckets) + 1, 16)
        ])
        self.cal_emb = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_cal_buckets, mask_token=None),
            tf.keras.layers.Embedding(len(unique_cal_buckets) + 1, 16)
        ])

        # --- MÜKEMMELLEŞTİRME: SBERT HUNİSİ (Daha Yumuşak Sıkıştırma) ---
        # 384 boyutu aniden 64'e düşürmek yerine 256 -> 128 ile yavaşça damıtıyoruz.
        self.sbert_dense = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation='relu'),
            tf.keras.layers.Dropout(0.3), # A100 çok güçlüdür, ezberlemeyi engellemek için dropout artırıldı
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dropout(0.1)
        ])

        # --- MÜKEMMELLEŞTİRME: BİRLEŞTİRME VE DENGELEME ---
        # Farklı kulelerden gelen bilgileri aynı ölçekte tutmak için Normalizasyon eklendi
        self.layer_norm = tf.keras.layers.LayerNormalization()

        # Tüm bilgileri harmanlayıp tek bir 128'lik vektör çıkaran son sinir ağı
        self.final_dense = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dense(EMBEDDING_DIM)
        ])

    def call(self, inputs):
        # 1. SBERT vektörünü sinir ağından geçir
        sbert_features = self.sbert_dense(inputs["sbert_vector"])

        # 2. Kategorik özellikleri ve SBERT'i yan yana yapıştır (Concat)
        concat_features = tf.concat([
            self.id_emb(inputs["recipe_id"]),
            self.time_emb(inputs["time_bucket"]),
            self.cal_emb(inputs["calorie_bucket"]),
            sbert_features
        ], axis=1)

        # 3. Dengeli Harmanlama (Layer Norm) ve Son Karar (Final Dense)
        x = self.layer_norm(concat_features)
        x = self.final_dense(x)

        return tf.math.l2_normalize(x, axis=1)

class FoodRecommender(tfrs.Model):
    def __init__(self):
        super().__init__()
        self.user_model = UserModel()
        self.recipe_model = RecipeModel()
        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=recipes_ds.map(self.recipe_model)
            )
        )

    def compute_loss(self, features, training=False):
        user_embeddings = self.user_model(features["user_id"])
        recipe_embeddings = self.recipe_model({
            "recipe_id": features["recipe_id"],
            "time_bucket": features["time_bucket"],
            "calorie_bucket": features["calorie_bucket"],
            "sbert_vector": features["sbert_vector"]
        })
        return self.task(user_embeddings, recipe_embeddings)

print("✅ Model mimarisi tanımlandı.")

🧠 Gelişmiş SBERT Destekli TFRS İki Kuleli Model Kuruluyor...
✅ Model mimarisi tanımlandı.


In [ ]:
# ==========================================
# 6. EĞİTİM, İNDEKSLEME VE "ISINMALI" KAYIT (CHECKPOINT & DİNAMİK LR)
# ==========================================


# Kayıt klasörümüz ve yeni "Auto-Save" (Checkpoint) klasörümüz
SAVE_DIR = '/content/drive/MyDrive/Tez3/SavedTFRS/'
CHECKPOINT_DIR = SAVE_DIR + 'checkpoints/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model = FoodRecommender()
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.05))

EPOCH_SAYISI = 100

# 1. SİGORTA: Erken Durdurma (Sabır 5'e çıkarıldı)
erken_durdurma = tf.keras.callbacks.EarlyStopping(
    monitor='loss',
    patience=5, # LR'nin düşüp toparlaması için 5 tur şans veriyoruz
    restore_best_weights=True
)

# 2. AKILLI FREN: Dinamik Öğrenme Oranı
dinamik_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,      # Model tıkanırsa hızı tam yarıya (%50) düşür
    patience=2,      # 2 tur boyunca kayıp düşmezse frene bas
    min_lr=0.0001,   # İnebileceği en düşük hız (Motor stop etmesin)
    verbose=1
)

# 3. AUTO-SAVE: Her Epoch'ta Drive'a Yedekleme (Hayat Kurtarır)
checkpoint_kayit = tf.keras.callbacks.ModelCheckpoint(
    filepath=CHECKPOINT_DIR + 'model_weights_epoch_{epoch:02d}.weights.h5',
    save_weights_only=True, # Sadece beynini (ağırlıklarını) kaydetmek çok hızlıdır
    save_best_only=False,   # Sadece en iyiyi değil, HER epoch'u kaydet
    verbose=1
)

print(f"\n🚀 DİNAMİK LR & CHECKPOINT İLE SBERT-TFRS Eğitimi Başlıyor ({EPOCH_SAYISI} Epoch)...\n")
print("⚠️ İnternet kopsa bile her epoch 'checkpoints' klasörüne yedeklenecektir.")

# Eğitime 3 callback'i (sigortayı) birden veriyoruz
model.fit(
    train_ds,
    epochs=EPOCH_SAYISI,
    callbacks=[erken_durdurma, dinamik_lr, checkpoint_kayit]
)

print("\n💾 Tahmin İndeksi Kuruluyor (BruteForce)...")
index = tfrs.layers.factorized_top_k.BruteForce(model.user_model, k=100)
index.index_from_dataset(
    tf.data.Dataset.zip((recipes_ds.map(lambda x: x["recipe_id"]), recipes_ds.map(model.recipe_model)))
)

# 🛡️ İŞTE HAYAT KURTARAN SİHİRLİ DOKUNUŞ (WARM-UP)
print("🔥 Model kaydetmeden önce ısınma turuna çıkarılıyor...")
_ = index(tf.constant(["dummy_user_123"]))

# === FİNAL KAYIT İŞLEMLERİ ===
tf.saved_model.save(index, SAVE_DIR + "tfrs_bruteforce_index")
sp.save_npz(SAVE_DIR + 'tfidf_matrix.npz', tfidf_matrix)

with open(SAVE_DIR + 'recipe_id_to_tfidf_idx.pkl', 'wb') as f:
    pickle.dump(recipe_id_to_tfidf_idx, f)

recipes.to_pickle(SAVE_DIR + 'recipes_processed.pkl')
train_df.to_pickle(SAVE_DIR + 'train_df.pkl')
test_df.to_pickle(SAVE_DIR + 'test_df.pkl')

print(f"✅ SBERT-TFRS Modeli ve Veriler Başarıyla {SAVE_DIR} Konumuna Kaydedildi!")


🚀 DİNAMİK LR & CHECKPOINT İLE SBERT-TFRS Eğitimi Başlıyor (100 Epoch)...

⚠️ İnternet kopsa bile her epoch 'checkpoints' klasörüne yedeklenecektir.
Epoch 1/100
316/316 [==============================] - ETA: 0s - factorized_top_k/top_1_categorical_accuracy: 0.0214 - factorized_top_k/top_5_categorical_accuracy: 0.0240 - factorized_top_k/top_10_categorical_accuracy: 0.0255 - factorized_top_k/top_50_categorical_accuracy: 0.0306 - factorized_top_k/top_100_categorical_accuracy: 0.0342 - loss: 15427.9099 - regularization_loss: 0.0000e+00 - total_loss: 15427.9099
Epoch 1: saving model to /content/drive/MyDrive/Tez3/SavedTFRS/checkpoints/model_weights_epoch_01.weights.h5
316/316 [==============================] - 381s 1s/step - factorized_top_k/top_1_categorical_accuracy: 0.0214 - factorized_top_k/top_5_categorical_accuracy: 0.0240 - factorized_top_k/top_10_categorical_accuracy: 0.0255 - factorized_top_k/top_50_categorical_accuracy: 0.0306 - factorized_top_k/top_100_categorical_accuracy: 0.03

✅ SBERT-TFRS Modeli ve Veriler Başarıyla /content/drive/MyDrive/Tez3/SavedTFRS/ Konumuna Kaydedildi!


In [ ]:
# ==========================================
# 7. METRİK DEĞERLENDİRME
# ==========================================
import gc
import numpy as np
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity
import tensorflow as tf

print("\n⚙️ 10.000 Kullanıcı İçin Kapsamlı Metrikler (Global AUC dahil) Hesaplanıyor...")

# Hızlı erişim için set dictionary yapıları
train_user_items = train_df.groupby("user_id")["recipe_id"].apply(set).to_dict()
test_user_items = test_df.groupby("user_id")["recipe_id"].apply(set).to_dict()

def tfrs_evaluate_global_metrics(index_model, test_users, train_history, test_history, tfidf_matrix, rec_to_tfidf, k=10, batch_size=256):
    precisions, recalls, aucs, hd_scores, diversities = [], [], [], [], []
    recommended_items = set()
    total_items = len(recipes)

    users_to_test = list(test_users.keys())
    # Her zaman aynı 10.000 kullanıcıyı test etmek için sabit Seed
    if len(users_to_test) > 10000:
        np.random.seed(42)
        users_to_test = np.random.choice(users_to_test, 10000, replace=False)

    for start in tqdm(range(0, len(users_to_test), batch_size), desc="TFRS Metrikleri"):
        batch_users = users_to_test[start:start+batch_size]

        # GLOBAL AUC İÇİN: Modelin tüm veritabanını taramasını istiyoruz (k=total_items)
        scores_batch, predicted_ids_batch = index_model(tf.constant(batch_users), k=total_items)

        for i, user_id in enumerate(batch_users):
            true_items = test_history.get(user_id, set())
            if not true_items: continue # Test setinde yemeği yoksa atla

            known_items = train_history.get(user_id, set())

            # GPU tensorlarını Python string ve numpy float'larına çevirme
            user_preds_raw = [pid.decode('utf-8') for pid in predicted_ids_batch[i].numpy()]
            user_scores_raw = scores_batch[i].numpy()

            valid_preds = []
            valid_scores = []

            # MÜKEMMELLEŞTİRME: Hızlı Maskeleme
            # Kullanıcının daha önce (train) yediği tarifleri tavsiyelerden çıkarıyoruz
            for pid, score in zip(user_preds_raw, user_scores_raw):
                if pid not in known_items:
                    valid_preds.append(pid)
                    valid_scores.append(score)

            if not valid_preds: continue # Filtrelemeden sonra önerecek bir şey kalmadıysa atla

            # --- 1. GLOBAL AUC ---
            # Modelin gerçekte yenen yemeği, diğer binlerce yemeğin ne kadar üstüne çıkardığını ölçer
            y_true_cand = [1 if pid in true_items else 0 for pid in valid_preds]
            if len(set(y_true_cand)) > 1: # AUC hesaplanabilmesi için en az 1 doğru, 1 yanlış olmalı
                try:
                    auc = roc_auc_score(y_true_cand, valid_scores)
                    aucs.append(auc)
                except ValueError: pass

            # --- 2. PRECISION, RECALL VE COVERAGE (İlk 10 Seçim) ---
            top_k_preds = valid_preds[:k]
            recommended_items.update(top_k_preds)

            hits = len(set(top_k_preds).intersection(true_items))
            precisions.append(hits / k)
            recalls.append(hits / len(true_items))

            # --- 3. HD95 VE DIVERSITY (Adil TF-IDF Cetveli İle Ölçüm) ---
            pred_tfidf_indices = [rec_to_tfidf[rid] for rid in top_k_preds if rid in rec_to_tfidf]
            true_tfidf_indices = [rec_to_tfidf[rid] for rid in true_items if rid in rec_to_tfidf]

            if len(pred_tfidf_indices) > 0 and len(true_tfidf_indices) > 0:
                pred_vecs = tfidf_matrix[pred_tfidf_indices]
                true_vecs = tfidf_matrix[true_tfidf_indices]

                # HD95 (Mesafe)
                d_p2t = np.min(euclidean_distances(pred_vecs, true_vecs), axis=1)
                d_t2p = np.min(euclidean_distances(true_vecs, pred_vecs), axis=1)
                hd95_val = max(np.percentile(d_p2t, 95), np.percentile(d_t2p, 95))
                hd_scores.append(hd95_val)

                # Diversity (Çeşitlilik)
                if len(pred_tfidf_indices) >= 2:
                    sim_matrix = cosine_similarity(pred_vecs)
                    upper = sim_matrix[np.triu_indices(len(pred_tfidf_indices), k=1)]
                    diversities.append(1 - np.mean(upper))

        # Her batch sonrası belleği temizle
        del scores_batch, predicted_ids_batch
        gc.collect()

    # Eğer test edilecek geçerli kullanıcı bulunamadıysa hataları engelle
    mean_p = np.mean(precisions) if precisions else 0
    mean_r = np.mean(recalls) if recalls else 0
    mean_auc = np.mean(aucs) if aucs else 0
    mean_hd = np.mean(hd_scores) if hd_scores else 0
    mean_div = np.mean(diversities) if diversities else 0
    cov = len(recommended_items) / total_items if total_items > 0 else 0

    return mean_p, mean_r, mean_auc, mean_hd, mean_div, cov

# Çıktıları Global AUC ile paketliyoruz
p, r, auc_global, hd95, div, cov = tfrs_evaluate_global_metrics(
    index_model=index, test_users=test_user_items, train_history=train_user_items,
    test_history=test_user_items, tfidf_matrix=tfidf_matrix, rec_to_tfidf=recipe_id_to_tfidf_idx, k=10
)

print("\n" + "="*45)
print(f"🎯 SBERT-TFRS TEZ SONUÇLARI")
print("="*45)
print(f"Precision@10 : {p:.4f}")
print(f"Recall@10    : {r:.4f}")
print(f"Global AUC   : {auc_global:.4f}")
print(f"HD95 (TF-IDF): {hd95:.4f}")
print(f"Diversity@10 : {div:.4f}")
print(f"Coverage     : {cov:.4f}")
print("="*45)


⚙️ 10.000 Kullanıcı İçin Kapsamlı Metrikler (Global AUC dahil) Hesaplanıyor...


TFRS Metrikleri: 100%|██████████| 40/40 [16:03<00:00, 24.08s/it]


🎯 SBERT-TFRS TEZ SONUÇLARI
Precision@10 : 0.0002
Recall@10    : 0.0007
Global AUC   : 0.6457
HD95 (TF-IDF): 1.3705
Diversity@10 : 0.9041
Coverage     : 0.4428


In [ ]:
# =============================================
# 🎯 SBERT-TFRS TEZ SONUÇLARI
# =============================================
# Precision@10 : 0.0002
# Recall@10    : 0.0007
# Global AUC   : 0.6457
# HD95 (TF-IDF): 1.3705
# Diversity@10 : 0.9041
# Coverage     : 0.4428
# =============================================

In [ ]:
# =============================================
# 🎯 SBERT-TFRS TEZ SONUÇLARI
# =============================================
# Precision@10 : 0.0002
# Recall@10    : 0.0004
# Global AUC   : 0.6626
# HD95 (TF-IDF): 1.3701
# Diversity@10 : 0.9024
# Coverage     : 0.4226
# =============================================